# Retail X — Final Demo and UI

This notebook launches the final Retail X submission in Google Colab.

The final repository is packaged so that the React UI is built once and then served by the FastAPI backend. This keeps the live demo on a single port and a single temporary Cloudflare URL.

**Flow:** upload ZIP → `uv sync` → build UI → `uv run server.py` → open one public demo URL.


In [ ]:
# Cell 1 — Upload and extract the final submission ZIP

from google.colab import files
from pathlib import Path
import zipfile
import shutil

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert zip_names, "Upload RetailX_Final_Submission.zip"

zip_path = Path("/content") / zip_names[0]
extract_root = Path("/content/retailx_final")

if extract_root.exists():
    shutil.rmtree(extract_root)

extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_root)

project_candidates = [
    p.parent
    for p in extract_root.rglob("pyproject.toml")
    if (p.parent / "server.py").exists()
]

assert project_candidates, "Could not locate the Retail X project folder."
PROJECT_DIR = project_candidates[0]

print("✅ Project extracted")
print("Project:", PROJECT_DIR)


In [ ]:
# Cell 2 — Configure OpenRouter secret

import os
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, (
    "Add OPENROUTER_API_KEY to Colab Secrets first. "
    "Create a key at https://openrouter.ai/settings/keys"
)

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ.setdefault(
    "OPENROUTER_MODEL",
    "google/gemini-3.1-flash-lite"
)

print("✅ OpenRouter secret loaded from Colab Secrets")
print("Model:", os.environ["OPENROUTER_MODEL"])


In [ ]:
# Cell 3 — Install uv + Node.js 22, sync Python dependencies, and build the UI

import os
import sys
import subprocess
from pathlib import Path

os.chdir(PROJECT_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "uv"],
    check=True
)

# Vite 8 requires a recent Node version.
subprocess.run(
    "curl -fsSL https://deb.nodesource.com/setup_22.x | bash -",
    shell=True,
    check=True
)
subprocess.run(
    ["apt-get", "install", "-y", "nodejs"],
    check=True
)

subprocess.run(["uv", "sync"], check=True)
subprocess.run(["npm", "install"], check=True)
subprocess.run(["npm", "run", "build"], check=True)

assert (PROJECT_DIR / "dist" / "index.html").exists()

print("✅ Python environment synchronized with uv")
print("✅ React interface built")


In [ ]:
# Cell 4 — Start Retail X with uv

import subprocess
import time
import requests
import os
from pathlib import Path

os.chdir(PROJECT_DIR)

if "server_process" in globals():
    try:
        if server_process.poll() is None:
            server_process.terminate()
            time.sleep(1)
    except Exception:
        pass

server_log_path = Path("/content/retailx_server.log")
server_log = open(server_log_path, "w")

server_env = os.environ.copy()

server_process = subprocess.Popen(
    ["uv", "run", "server.py"],
    cwd=str(PROJECT_DIR),
    env=server_env,
    stdout=server_log,
    stderr=subprocess.STDOUT
)

health = None

for _ in range(90):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/api/health",
            timeout=3
        )
        if response.status_code == 200:
            health = response.json()
            break
    except Exception:
        pass

    if server_process.poll() is not None:
        break

    time.sleep(1)

if health is None:
    server_log.flush()
    print(server_log_path.read_text(encoding="utf-8", errors="ignore")[-5000:])
    raise RuntimeError("Retail X server did not start.")

print("✅ Retail X is running")
print(health)


In [ ]:
# Cell 5 — Verify a real pending demo case

import requests

demo = requests.get(
    "http://127.0.0.1:8000/api/demo-case",
    timeout=10
)

print("Status:", demo.status_code)
print(demo.json())


In [ ]:
# Cell 6 — Create one temporary public demo URL

import subprocess
import time
import re
from pathlib import Path

cloudflared = Path("/usr/local/bin/cloudflared")

if not cloudflared.exists():
    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O",
            str(cloudflared)
        ],
        check=True
    )
    subprocess.run(["chmod", "+x", str(cloudflared)], check=True)

if "demo_tunnel" in globals():
    try:
        if demo_tunnel.poll() is None:
            demo_tunnel.terminate()
            time.sleep(1)
    except Exception:
        pass

tunnel_log_path = Path("/content/retailx_tunnel.log")
tunnel_log = open(tunnel_log_path, "w")

demo_tunnel = subprocess.Popen(
    [
        str(cloudflared),
        "tunnel",
        "--url",
        "http://127.0.0.1:8000",
        "--no-autoupdate"
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

DEMO_URL = None

for _ in range(60):
    time.sleep(1)
    log_text = tunnel_log_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )
    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        log_text
    )
    if match:
        DEMO_URL = match.group(0)
        break

assert DEMO_URL, "Cloudflare tunnel URL was not created."

print("=" * 70)
print("🚀 RETAIL X DEMO READY")
print("=" * 70)
print(DEMO_URL)
print()
print("Keep this Colab runtime running while using the demo.")


## Live demo script

1. Open the URL printed above.
2. The interface loads a real pending τ²-Bench order.
3. Press **Send**.
4. Show the workflow:
   **Identity Verification → Retrieve Order → Policy Check → Decision → Human Approval**
5. Press **Approve cancellation**.
6. Show the order status changing to **cancelled**.

The public URL is temporary by design. The repository itself runs locally with `uv run server.py` after the frontend build.
